Run 'pip install --upgrade -r "requirements.txt"' before running

Imports

In [1]:
import json
import requests

import pandas as pd
import numpy as np

import string
from collections import Counter

from gensim import corpora, models

## Functions for data manipulation

Get stop words

In [2]:
stop_en = requests.get('https://data.analytics.hathitrust.org/data/text/default_stopwords_en.txt')
stop_en = stop_en.text
stop_en = stop_en.split()

stop_set = set(s.lower() for s in stop_en)
custom_stops = {"'s", "''", "``", "--", "p.", "-RRB-", "-LRB-", "...", "n't", "ing", "Mr.", "Mrs.", "3", "2", "1", "¬", "B."}
stop_set.update(custom_stops)
stop_set.update(list(string.punctuation))
stop_set = {s.lower() for s in stop_set}

Functions to get volume data

In [3]:
def retrieve_data(volume_id):
    res = requests.get("https://data.htrc.illinois.edu/ef-api/volumes/" + volume_id)
    raw = res.content

    if (res.status_code == 200):
        try:
            parsed = json.loads(raw.decode("utf-8"))
            return parsed
        except Exception as e:
            # print("Failed to parse JSON response")
            return "ERROR"
    if (res.status_code == 404):
        return "NOT FOUND"

Get bag-of-words for full volume instead of page

In [4]:
def tokens_count(record):
    word_key_count = Counter()
    
    for page in record['data']['features']['pages']:
        if page['body']:
            for token, pos_count in page['body']['tokenPosCount'].items():
                word_key_count[token] += sum(pos_count.values())
                    
    return word_key_count

LDA topic model

In [5]:
def get_keywords(texts):

    all_tokens = [list(d.keys()) for d in texts]
    dictionary = corpora.Dictionary(all_tokens)

    corpus = []
    for d in texts:
        bow = [(dictionary.token2id[word], count) for word, count in d.items() if word in dictionary.token2id]
        corpus.append(bow)

    num_topics = 30
    passes = 50
    iterations = 200

    lda_model = models.LdaModel(
        corpus=corpus, 
        id2word=dictionary, 
        num_topics=num_topics, 
        passes=passes, 
        iterations=iterations,
        alpha='auto', 
        eta='auto',
        random_state=42
    )
    
    topics = lda_model.show_topics(num_topics=5, num_words=20, formatted=False)    
    return topics

Clean data

In [6]:
def clean_df(data, stop_sey):
    df = pd.DataFrame({'count': data})
    df = df.reset_index().rename(columns={'index': 'word'})
    df = df[~df['word'].str.lower().isin(stop_set)]
    df = df[df['word'].str.len() > 2]
    return df

## Getting Data

Get all workset data

Current wsid = f4c577e0-3f33-11f1-806f-2d821d8659ab

In [7]:
import requests
wsid = "f4c577e0-3f33-11f1-806f-2d821d8659ab"
workset = requests.get(f"https://worksets.hathitrust.org/wsid/{wsid}")

workset_content = workset.content
raw = json.loads(workset_content)

Get each volume's data

In [8]:
data = raw['gathers']

readable_json = json.dumps(data, indent=4) # Indent with 4 spaces

x = readable_json.split()
y = "http://hdl.handle.net/2027/"
volumes = []
for i in x:
    if y in i:
        z = i.replace(y,"")
        z = z.replace('"',"")
        z = z.replace(":", "+")
        z = z.replace("\\\\", "=")
        volumes.append(z)

print(len(volumes))

788


In [9]:
worked = 0
failed = 0
noget = 0

volume_data = {}

for vol in volumes:
    data = retrieve_data(vol)
    print(vol)

    if (data == "NOT FOUND"): 
        noget += 1
    elif (data == "ERROR"):
        failed += 1
    else: 
        worked += 1
        volume_data[vol] = data

print ("WORKED: " + str(worked))
print ("FAILED: " + str(failed))
print ("NOT FOUND: " + str(noget))

print(len(volume_data))

ucbk.ark+/28722/h2vx06f8c
ucbk.ark+/28722/h2gm8215d
ucbk.ark+/28722/h2s46hj5q
ucbk.ark+/28722/h2hm52x57
ucbk.ark+/28722/h2dv1d127
ucbk.ark+/28722/h21j97m29
ucbk.ark+/28722/h22j68h0k
ucbk.ark+/28722/h2pc2tn1t
ucbk.ark+/28722/h2q815471
ucbk.ark+/28722/h2ft8dx30
ucbk.ark+/28722/h2794163g
ucbk.ark+/28722/h2zs2kr5t
ucbk.ark+/28722/h2w08wv1p
ucbk.ark+/28722/h2gt5ft2r
ucbk.ark+/28722/h2d21rw91
ucbk.ark+/28722/h20r9mg6r
ucbk.ark+/28722/h25h7c60r
ucbk.ark+/28722/h2s756x62
ucbk.ark+/28722/h2z02zn0p
ucbk.ark+/28722/h2jq0t68t
ucbk.ark+/28722/h2kp7v390
ucbk.ark+/28722/h2b56dh2w
ucbk.ark+/28722/h2c53fd05
ucbk.ark+/28722/h20000c21
ucbk.ark+/28722/h20v89v8k
ucbk.ark+/28722/h2rf5kt3k
ucbk.ark+/28722/h2x34n44n
ucbk.ark+/28722/h2hx1633r
ucbk.ark+/28722/h2pn8xt0v
ucbk.ark+/28722/h29c6sc7t
ucbk.ark+/28722/h2kw57w6q
ucbk.ark+/28722/h26m33g39
ucbk.ark+/28722/h27m04c22
ucbk.ark+/28722/h2028pr5d
ucbk.ark+/28722/h28k75837
ucbk.ark+/28722/h2rj49669
ucbk.ark+/28722/h2sj1b371
ucbk.ark+/28722/h2ns0m93w
ucbk.ark+/28

Get each unique location

In [10]:
locations = {}


for key in volume_data:
    try:
        loc = volume_data[key]['data']['metadata']['pubPlace']['name']
        if loc not in locations:
            locations[loc] = []
        locations[loc].append(key)
    except Exception as e:
        continue

Get all bag-of-words data for each document into dict with documents per state

In [11]:
data_per_state = {}
for loc, documents in locations.items():
    data_per_state[loc] = []
    for doc in documents:
        parsed = retrieve_data(doc)
        counts = tokens_count(parsed)
        df = clean_df(counts, stop_en)
        dict_ = df.set_index('word')['count'].to_dict()
        data_per_state[loc].append(dict_)

## Get Keywords

In [12]:
keywords_per_state = {}
for loc, documents in data_per_state.items():
    keywords_per_state[loc] = get_keywords(documents)

### Prints all top topics clusters

In [13]:
for loc, words in keywords_per_state.items():
    print(loc)
    print(words)

Connecticut
[(np.int64(20), [('Yale', np.float32(2.3816103e-05)), ('Haven', np.float32(2.3812334e-05)), ('students', np.float32(2.3807488e-05)), ('time', np.float32(2.380641e-05)), ('Bernie', np.float32(2.380616e-05)), ('man', np.float32(2.3805416e-05)), ('years', np.float32(2.3805136e-05)), ('back', np.float32(2.3804912e-05)), ('good', np.float32(2.3804268e-05)), ('people', np.float32(2.3804254e-05)), ('year', np.float32(2.380424e-05)), ('Wilson', np.float32(2.3803961e-05)), ('city', np.float32(2.3803937e-05)), ('life', np.float32(2.3803841e-05)), ('news', np.float32(2.3803635e-05)), ('Register', np.float32(2.3803444e-05)), ('young', np.float32(2.3803392e-05)), ('school', np.float32(2.3803219e-05)), ('work', np.float32(2.380303e-05)), ('morning', np.float32(2.380283e-05))]), (np.int64(1), [('Yale', np.float32(2.3807006e-05)), ('students', np.float32(2.3804982e-05)), ('time', np.float32(2.3804954e-05)), ('man', np.float32(2.38049e-05)), ('people', np.float32(2.3804234e-05)), ('American

In [14]:
print(keywords_per_state)

{'Connecticut': [(np.int64(20), [('Yale', np.float32(2.3816103e-05)), ('Haven', np.float32(2.3812334e-05)), ('students', np.float32(2.3807488e-05)), ('time', np.float32(2.380641e-05)), ('Bernie', np.float32(2.380616e-05)), ('man', np.float32(2.3805416e-05)), ('years', np.float32(2.3805136e-05)), ('back', np.float32(2.3804912e-05)), ('good', np.float32(2.3804268e-05)), ('people', np.float32(2.3804254e-05)), ('year', np.float32(2.380424e-05)), ('Wilson', np.float32(2.3803961e-05)), ('city', np.float32(2.3803937e-05)), ('life', np.float32(2.3803841e-05)), ('news', np.float32(2.3803635e-05)), ('Register', np.float32(2.3803444e-05)), ('young', np.float32(2.3803392e-05)), ('school', np.float32(2.3803219e-05)), ('work', np.float32(2.380303e-05)), ('morning', np.float32(2.380283e-05))]), (np.int64(1), [('Yale', np.float32(2.3807006e-05)), ('students', np.float32(2.3804982e-05)), ('time', np.float32(2.3804954e-05)), ('man', np.float32(2.38049e-05)), ('people', np.float32(2.3804234e-05)), ('Amer

In [15]:
loc_topics = {}
for loc, topics in keywords_per_state.items():
    sorted_topics = sorted(topics, key=lambda x: x[0])
    topics_dict = {}
    
    for i in sorted_topics:
        topic_words = {}
        for j in i[1]:
            topic_words[j[0]] = j[1]
        topics_dict[i[0]] = topic_words
    loc_topics[loc] = topics_dict

In [16]:
for loc, topics in loc_topics.items():
    print(loc)
    for topic_id, topic_words in topics.items():
        print(f"___ {topic_id} ___")
        for word, prob in topic_words.items():
            print(f"\t{word}:{prob}")
    print()

Connecticut
___ 1 ___
	Yale:2.380700607318431e-05
	students:2.3804981537978165e-05
	time:2.380495425313711e-05
	man:2.3804899683455005e-05
	people:2.3804233933333308e-05
	American:2.380305522819981e-05
	Haven:2.380296245974023e-05
	young:2.3802800569683313e-05
	good:2.38025368162198e-05
	world:2.380237310717348e-05
	war:2.3802202122169547e-05
	life:2.3802052965038456e-05
	history:2.380179830652196e-05
	years:2.3801765564712696e-05
	Vietnam:2.3801714633009396e-05
	year:2.3801627321518026e-05
	student:2.380154182901606e-05
	tion:2.3801314455340616e-05
	make:2.380127989454195e-05
	Weaver:2.380123441980686e-05
___ 6 ___
	Yale:0.0072954436764121056
	people:0.004999694414436817
	time:0.0031483047641813755
	man:0.0026937208604067564
	Haven:0.0025686961598694324
	work:0.002268324140459299
	years:0.002084259409457445
	made:0.002069206442683935
	student:0.002051200484856963
	students:0.002039156621322036
	play:0.002019667997956276
	back:0.0020056795328855515
	life:0.001945418305695057
	American:

In [17]:
import json

json_output = {}

for loc, topics in loc_topics.items():
    json_output[loc] = []
    for topic_id, topic_words in topics.items():
        topic_entry = {
            "topic_id": int(topic_id),
            "keywords": [
                {"word": word, "probability": float(prob)} 
                for word, prob in topic_words.items()
            ]
        }
        json_output[loc].append(topic_entry)

file_name = "output_test.json"
with open(file_name, 'w') as f:
    json.dump(json_output, f, indent=4)

print(f"Data successfully saved to {file_name}")

Data successfully saved to output_test.json
